# [LAB-04] 5. 데이터 병합

## #01. 준비작업

### 1. 패키지 참조

In [1]:
from jussam import load_data
from pandas import DataFrame
from pandas import merge, concat

📦 연세대학교 주영아 교수가 제작한 라이브러리를 사용중입니다.
📧 Email(1): j.purplerose@yonsei.ac.kr
📧 Email(2): j.purplerose@gmail.com
📝 Website: https://juyounga.kr/


### 2. 첫 번째 샘플 데이터 가져오기

In [2]:
고객 = load_data("customer_no_name")
고객

📚 데이터프레임 병합 실습을 위한 고객번호와 고객 이름에 대한 가상 데이터(인덱스와 메타데이터 없음)


,고객번호,이름
0,1001,둘리
1,1002,도우너
2,1003,또치
3,1004,길동
4,1005,희동
5,1006,마이콜
6,1007,영희


### 3. 두 번째 샘플 데이터 가져오기

In [3]:
매출 = load_data("customer_no_pay")
매출

📚 데이터프레임 병합 실습을 위한 고객번호와 매출액에 대한 가상 데이터(인덱스와 메타데이터 없음)


,고객번호,금액
0,1001,10000
1,1001,20000
2,1005,15000
3,1006,5000
4,1008,100000
5,1001,30000


## #02. 열 기준 병합

### 1. 일치하는 데이터끼의 병합

- SQL의 INNER JOIN과 동일
- 두개의 데이터 프레임에서 이름이 동일한 컬럼을 기준으로 같은 데이터끼리 병합하고 일치하지 않는 데이터는 버려진다.
- 만약 양쪽 데이터프레임의 공통 컬럼에 중복 데이터가 여러 개 있는 경우는 모든 경우의 수를 따져서 조합을 만들어 낸다.(예: 1001번 둘리의 데이터)

In [4]:
merge(고객, 매출)

,고객번호,이름,금액
0,1001,둘리,10000
1,1001,둘리,20000
2,1001,둘리,30000
3,1005,희동,15000
4,1006,마이콜,5000


### 2. 왼쪽 데이터 프레임을 기준으로 병합

- SQL의 LEFT OUTER JOIN과 동일
- 고객(왼쪽) 데이터를 기준으로 일치하는 매출 데이터를 병합한다.
  - 고객 데이터의 모든 항목에 대한 출력이 보장된다.
  - 고객 데이터에 매칭되는 매출 데이터가 없는 경우 빈 값으로 처리된다.

In [5]:
merge(고객, 매출, how="left")

,고객번호,이름,금액
0,1001,둘리,10000.000
1,1001,둘리,20000.000
2,1001,둘리,30000.000
3,1002,도우너,NaN
4,1003,또치,NaN
5,1004,길동,NaN
6,1005,희동,15000.000
7,1006,마이콜,5000.000
8,1007,영희,NaN


### 3. 오른쪽 데이터 프레임을 기준으로 병합

- SQL의 RIGHT OUTER JOIN과 동일

In [6]:
merge(고객, 매출, how="right")

,고객번호,이름,금액
0,1001,둘리,10000
1,1001,둘리,20000
2,1005,희동,15000
3,1006,마이콜,5000
4,1008,NaN,100000
5,1001,둘리,30000


### 4. 모든 데이터의 교차 병합 (합집합)

In [7]:
merge(고객, 매출, how="outer")

,고객번호,이름,금액
0,1001,둘리,10000.000
1,1001,둘리,20000.000
2,1001,둘리,30000.000
3,1002,도우너,NaN
4,1003,또치,NaN
5,1004,길동,NaN
6,1005,희동,15000.000
7,1006,마이콜,5000.000
8,1007,영희,NaN
9,1008,NaN,100000.000


## #03. 병합 기준 열 지정하기

### 1. 샘플 데이터 가져오기

In [8]:
cd1 = load_data("customer_data1")
cd1

📚 데이터 병합용 실습 데이터 (1) (인덱스/메타데이터 없음)


,고객명,날짜,데이터
0,민수,2018-01-01,20000
1,수영,2018-01-01,100000


In [9]:
cd2 = load_data("customer_data2")
cd2

📚 데이터 병합용 실습 데이터 (2) (인덱스/메타데이터 없음)


,고객명,데이터
0,민수,21세
1,수영,20세


### 2. 기본 병합

- 두 데이터 프레임에서 이름이 같은 열은 모두 키가 된다.
- 샘플 데이터에서는 `데이터`라는 이름의 변수가 `cd1`은 `int`, `cd2`는 `str`타입이므로 병합 기준을 충족하지 않는다.
- **그러므로 아래 코드는 에러**

In [10]:
# merge(cd1, cd2)

```
ValueError: You are trying to merge on int64 and object columns for key '데이터'. If you wish to proceed you should use pd.concat
```

### 3. 병합 기준 설정

- `on` 파라미터로 병합의 기준이 되는 열을 지정해야 한다.
- 병합 기준 열이 아니면서 이름이 같은 열에는 `_x` 또는 `_y` 와 같은 접미사가 붙는다.

In [11]:
tmp = merge(cd1, cd2, on='고객명')
tmp

,고객명,날짜,데이터_x,데이터_y
0,민수,2018-01-01,20000,21세
1,수영,2018-01-01,100000,20세


### 4. 두 데이터 프레임의 모든 컬럼 이름이 다른 경우

왼쪽의 기준열 이름과 오른쪽의 기준열 이름을 각각 설정해야 한다.

In [12]:
df_left = DataFrame({'이름': ['영희', '철수'], '국어': [87, 91]})
df_left

,이름,국어
0,영희,87
1,철수,91


In [13]:
df_right = DataFrame({'성명': ['영희', '철수'], '영어': [90, 82]})
df_right

,성명,영어
0,영희,90
1,철수,82


In [14]:
r3 = merge(df_left, df_right, left_on=['이름'], right_on=['성명'])
r3

,이름,국어,성명,영어
0,영희,87,영희,90
1,철수,91,철수,82


## #04. 인덱스를 활용한 병합

### 1. 학생의 이름을 인덱스로 갖는 두 데이터 프레임

In [15]:
df_left = DataFrame({'수학': [90, 82]}, index=['민철', '봉구'])
df_left

,수학
민철,90
봉구,82


In [16]:
df_right = DataFrame({'국어': [90, 82]}, index=['민철', '철수'])
df_right

,국어
민철,90
철수,82


### 2. 병합조건에서 index를 사용하도록 지정하여 Inner Join 하기

In [17]:
merge(df_left, df_right, left_index=True, right_index=True)

,수학,국어
민철,90,90


### 3. 병합조건에서 index를 사용하도록 지정 Left Outer Join 하기

In [18]:
merge(df_left, df_right, left_index=True, right_index=True,
      how="left")

,수학,국어
민철,90,90.000
봉구,82,NaN


### 4. 병합조건에서 index를 사용하도록 지정 Right Outer Join 하기

In [19]:
merge(df_left, df_right, left_index=True, right_index=True,
      how="right")

,수학,국어
민철,90.000,90
철수,NaN,82


### 5. 병합조건에서 index를 사용하도록 지정 Full Outer Join 하기

In [20]:
merge(df_left, df_right, left_index=True, right_index=True,
      how="outer")

,수학,국어
민철,90.000,90.000
봉구,82.000,NaN
철수,NaN,82.000


## #05. 인덱스와 컬럼을 각각 기준으로 하기

### 1. 샘플 데이터 프레임 생성

In [21]:
df_left = DataFrame({'수학': [90, 82]}, index=['민철', '봉구'])
df_left

,수학
민철,90
봉구,82


In [22]:
df_right = DataFrame({'성명': ['민철', '철수'], '영어': [90, 82]})
df_right

,성명,영어
0,민철,90
1,철수,82


### 2. 왼쪽은 인덱스, 오른쪽은 컬럼을 기준으로 Inner Join

In [23]:
merge(df_left, df_right, left_index=True, right_on=['성명'])

,수학,성명,영어
0,90,민철,90


### 3. 왼쪽은 인덱스, 오른쪽은 컬럼을 기준으로 Left Outer Join

In [24]:
merge(df_left, df_right, left_index=True, right_on=['성명'],
      how='left')

,수학,성명,영어
0.000,90,민철,90.000
NaN,82,봉구,NaN


### 4. 왼쪽은 인덱스, 오른쪽은 컬럼을 기준으로 Right Outer Join

In [25]:
merge(df_left, df_right, left_index=True, right_on=['성명'], how='right')

,수학,성명,영어
0,90.000,민철,90
1,NaN,철수,82


### 5. 왼쪽은 인덱스, 오른쪽은 컬럼을 기준으로 Full Outer Join

In [26]:
tmp = merge(df_left, df_right, left_index=True, right_on=['성명'],
            how='outer')
tmp

,수학,성명,영어
0.000,90.000,민철,90.000
NaN,82.000,봉구,NaN
1.000,NaN,철수,82.000


### 6. 병합 후에는 인덱스를 정리해야 한다.

In [27]:
tmp2 = tmp.set_index('성명')
tmp2

,수학,영어
성명,,
민철,90.000,90.000
봉구,82.000,NaN
철수,NaN,82.000


## #06. 행 기준 병합

### 1. 샘플 데이터 생성

In [28]:
df1 = DataFrame({'이름': ['영희', '철수'], '국어': [87, 91]})
df1

,이름,국어
0,영희,87
1,철수,91


In [29]:
df2 = DataFrame({'이름': ['민철', '수현'], '국어': [78, 92]})
df2

,이름,국어
0,민철,78
1,수현,92


### 2. concat() 함수 기본 사용 방법

- 병합할 데이터 프레임을 리스트로 묶는다. (2개 이상 가능)
- 각 데이터프레임이 갖는 인덱스는 그대로 유지된다.(인덱스 중복 발생)

In [30]:
concat([df1, df2])

,이름,국어
0,영희,87
1,철수,91
0,민철,78
1,수현,92


### 3. 병합 결과에서 index를 재구성하기 (1)

- 지금까지 공부한 내용만으로 처리하기

In [31]:
인덱스_재구성df = concat([df1, df2])
인덱스_재구성df.reset_index(inplace=True, drop=True)
인덱스_재구성df

,이름,국어
0,영희,87
1,철수,91
2,민철,78
3,수현,92


### 4. 병합 결과에서 index를 재구성하기 (2)

- `concat()` 함수에 `ignore_index=True` 파라미터를 설정하면 병합 후 인덱스를 재구성 한다.

In [32]:
concat([df1, df2], ignore_index=True)

,이름,국어
0,영희,87
1,철수,91
2,민철,78
3,수현,92


## #07. 서로 다른 변수를 갖는 데이터 프레임의 병합

### 1. 샘플 데이터 구성

In [33]:
df3 = DataFrame({'이름': ['영희', '철수'], '국어': [87, 91]})
df3

,이름,국어
0,영희,87
1,철수,91


In [34]:
df4 = DataFrame({'이름': ['민철', '수현'], '수학': [78, 92]})
df4

,이름,수학
0,민철,78
1,수현,92


### 2. 데이터 병합

- 일치하지 않는 부분에 대해서는 `NaN`으로 처리한다.

In [35]:
concat([df3, df4], ignore_index=True)

,이름,국어,수학
0,영희,87.000,NaN
1,철수,91.000,NaN
2,민철,NaN,78.000
3,수현,NaN,92.000


## #08. 인덱스를 갖는 데이터 프레임의 병합

### 1. 샘플 데이터 구성

In [36]:
df5 = DataFrame({'국어': [87, 91, 85]},
                index=['영희', '철수', '민수'])
df5

,국어
영희,87
철수,91
민수,85


In [37]:
df6 = DataFrame({'국어': [78, 92, 82]},
                index=['민철', '수현', '민수'])
df6

,국어
민철,78
수현,92
민수,82


### 2. 데이터 병합

- 인덱스도 함께 병합된다.
- 각 데이터 프레임간에 중복되는 인덱스가 존재할 경우 각 행은 개별적으로 존재한다. (인덱스 중복)

In [38]:
concat([df5, df6])

,국어
영희,87
철수,91
민수,85
민철,78
수현,92
민수,82


### 3. 인덱스를 제외하고 병합하기

`ignore_index=True`라는 파라미터를 설정한다.

In [39]:
concat([df5, df6], ignore_index=True)

,국어
0,87
1,91
2,85
3,78
4,92
5,82
